In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
import joblib

features = [
    'Age', 'Gender', 'Socio-economic Status', 'Insurance', 'Hospital Type',
    'Body Temperature', 'SpO2', 'Respiratory Rate', 'Heart Rate', 'BMI',
    'Haemoglobin', 'Calcium', 'Phosphorus', 'Platelet Count', 'Cholesterol',
    'Sodium', 'Potassium', 'Neutrophils', 'Lymphocytes', 'CRP', 'WBC Count'
]

df = pd.read_csv("INDIA_HEALTHCARE_DATA_2026.csv").dropna(subset=features + ['Diseases'])

X = df[features].copy()
y = df['Diseases']

print("Encoding features")
encoders = {}
for col in ['Gender', 'Socio-economic Status', 'Insurance', 'Hospital Type']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("Training XGBoost Model")
model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(target_encoder.classes_),
    eval_metric='mlogloss',
    learning_rate=0.1,
    max_depth=6,
    n_estimators=200,
    random_state=42
)
model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred = model.predict(X_test)
print("\n--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Weighted Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Weighted F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")

joblib.dump(model, 'xgboost_triage.pkl')
joblib.dump(encoders, 'feature_encoders.pkl')
joblib.dump(target_encoder, 'target_encoder.pkl')
print("\nModel and encoders saved successfully!")

Encoding features
Training XGBoost Model

--- Model Performance ---
Accuracy: 0.9776
Weighted Recall: 0.9776
Weighted F1-Score: 0.9775

Model and encoders saved successfully!


In [ ]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load('xgboost_triage.pkl')
encoders = joblib.load('feature_encoders.pkl')
target_encoder = joblib.load('target_encoder.pkl')

critical_patient_data = {
    'Age': 45,
    'Gender': 'Male',
    'Socio-economic Status': 'Low',
    'Insurance': 'Not-Covered',
    'Hospital Type': 'Government',
    'Body Temperature': 104.0,
    'SpO2': 95.0,
    'Respiratory Rate': 22.0,
    'Heart Rate': 130.0,
    'BMI': 24.5,
    'Haemoglobin': 14.0,
    'Calcium': 8.5,
    'Phosphorus': 3.0,
    'Platelet Count': 20.0,
    'Cholesterol': 150.0,
    'Sodium': 135.0,
    'Potassium': 3.8,
    'Neutrophils': 80.0,
    'Lymphocytes': 15.0,
    'CRP': 25.0,
    'WBC Count': 2.5
}

df_critical = pd.DataFrame([critical_patient_data])

for col in ['Gender', 'Socio-economic Status', 'Insurance', 'Hospital Type']:
    df_critical[col] = encoders[col].transform(df_critical[col])

predicted_probs = model.predict_proba(df_critical)[0]
top_3_indices = np.argsort(predicted_probs)[-3:][::-1]

print(" CRITICAL PATIENT TRIAGE RESULTS \n")
for i, idx in enumerate(top_3_indices):
    disease = target_encoder.classes_[idx]
    confidence = predicted_probs[idx] * 100
    print(f"Priority {i+1}: {disease} (Confidence: {confidence:.2f}%)")

if target_encoder.classes_[top_3_indices[0]] == 'Dengue':
    print("\n SUCCESS: The model correctly identified the critical Dengue case.")
else:
    print("\n WARNING: The model failed to prioritize the critical Dengue case.")

 CRITICAL PATIENT TRIAGE RESULTS 

Priority 1: Dengue (Confidence: 99.99%)
Priority 2: Thyroid Disorder (Confidence: 0.01%)
Priority 3: Hypertension (Confidence: 0.00%)

 SUCCESS: The model correctly identified the critical Dengue case.


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
import time

features = [
    'Age', 'Gender', 'Socio-economic Status', 'Insurance', 'Hospital Type',
    'Body Temperature', 'SpO2', 'Respiratory Rate', 'Heart Rate', 'BMI',
    'Haemoglobin', 'Calcium', 'Phosphorus', 'Platelet Count', 'Cholesterol',
    'Sodium', 'Potassium', 'Neutrophils', 'Lymphocytes', 'CRP', 'WBC Count'
]

print("Loading data...")
df = pd.read_csv("INDIA_HEALTHCARE_DATA_2026.csv").dropna(subset=features + ['Diseases'])

X = df[features].copy()
y = df['Diseases']

for col in ['Gender', 'Socio-economic Status', 'Insurance', 'Hospital Type']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(target_encoder.classes_),
    eval_metric='mlogloss',
    random_state=42,
    tree_method='hist',
    device='cuda'
)


grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    verbose=2,
    n_jobs=-1
)

print("Starting Grid Search... This may take a few minutes depending on your hardware.")
start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()


print("\n--- Tuning Complete ---")
print(f"Time Elapsed: {(end_time - start_time) / 60:.2f} minutes")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")
print("\nBest Parameters Found:")
for param, value in grid_search.best_params_.items():
    print(f" - {param}: {value}")

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"\nFinal Accuracy on Hold-Out Test Set: {test_accuracy:.4f}")

Loading data...
Starting Grid Search... This may take a few minutes depending on your hardware.
Fitting 3 folds for each of 36 candidates, totalling 108 fits


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [04:39:24] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [04:39:24] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)



--- Tuning Complete ---
Time Elapsed: 10.04 minutes
Best Cross-Validation Accuracy: 0.9767

Best Parameters Found:
 - learning_rate: 0.1
 - max_depth: 4
 - n_estimators: 100
 - subsample: 0.8

Final Accuracy on Hold-Out Test Set: 0.9770
